<img src="https://github.com/nicholasmetherall/digital-earth-pacific-macblue-activities/blob/main/attachments/images/DE_Pacific_banner.JPG?raw=true" width="900"/>

Figure 1.1.a. Jupyter environment + Python notebooks

# Digital Earth Pacific Notebook 1 prepare postcard and load data to csv

The objective of this notebook is to prepare a geomad postcard for your AOI (masking, scaling and loading additional band ratios and spectral indices) and sampling all the datasets into a csv based on your training data geodataframe.

This notebook focuses on generating postcard CSV dataset for machine learning model training. Throughout the workflow, we will load and preprocess spatial training data, extract satellite image values from each training point, calculate spectral indices, apply masking, and export the final training dataset as a CSV file that can later be used for Random Forest classification.

In [1]:
# # This cell is for papermill parameters. DO NOT CHANGE THE VARIABLE NAMES.
# # Default values for manual execution (papermill will override these)
# input_geojson_path = None
# output_csv_path = None

## Step 1.1: Configure the environment
In this cell, we are importing the main Python libraries required for the postcard generation workflow. These libraries will allow us to work with spatial vector data, coordinate reference systems, satellite imagery, numerical arrays, and interactive mapping tools. Together, they provide the foundation for loading training data, processing imagery, and extracting spectral values for machine learning.

In [2]:
from odc.geo import Geometry
from odc.stac import load
from odc.geo.xr import write_cog
import rasterio as rio
import os
from datetime import datetime
from shapely.geometry import Polygon
from shapely import box
from pyproj import CRS 
import folium
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio as rio
import xarray as xr
import rioxarray
from ipyleaflet import basemaps
from numpy.lib.stride_tricks import sliding_window_view
import pystac_client
from dask.distributed import Client as DaskClient
from odc.stac import load, configure_s3_access
import planetary_computer
# from odc.stac import load
from pystac.client import Client
from skimage.feature import graycomatrix, graycoprops
from utils import load_data, load_s1_dem, scale, calculate_band_indices, apply_mask, mask_water, all_masks, do_prediction

This section demonstrates how automatic reloading can be enabled within the notebook environment. Participants learn how this improves workflow efficiency during development by automatically updating imported scripts whenever changes are made externally, removing the need to repeatedly restart the notebook kernel.

>%load_ext autoreload

>%autoreload 2

In [3]:
%load_ext autoreload

%autoreload 2

In this step, participants define project-specific variables such as user initials, study site names and date. These variables are later used to create standardized filenames and organize outputs consistently throughout the workflow. This helps establish good data management and reproducibility practices.
###### Enter your initials
>initials = " "

###### Enter your site name
>site = " "

In this line we are using the datetime.now to automatically adjust the datetime to the time we run this notebook.
###### Date
>date = datetime.now()

In this line, participants will generate a standardized version name for the postcard dataset using project-specific variables and the current date. The code combines the user initials, study site name, and formatted date into a single descriptive filename that can be used to organize outputs consistently throughout the workflow.

>version = f"{initials}-{site}-{date.strftime('%d%m%Y')}_postcard_4"

>print(version)

In [4]:
initials = "nm"

site = "nadroga"

This exercise step introduces participants to loading and organizing multiple GeoJSON training datasets. The workflow searches through the training-data directory, identifies all valid GeoJSON files, and prepares them for processing. Participants learn how datasets from multiple sources can be combined into a unified training dataset for machine learning.

In [5]:
gdfs = []
postcards_path = "training-data/"
file_extension: str = ".geojson"

for filename in os.listdir(postcards_path):
    file_path = os.path.join(postcards_path, filename)
    if os.path.isfile(file_path) and filename.endswith(file_extension):
    # try:
        gdf = gpd.read_file(file_path)
        gdfs.append(gdf)

In [6]:
for filename in os.listdir(postcards_path):
    file_path = os.path.join(postcards_path, filename)
    if os.path.isfile(file_path) and filename.endswith(file_extension):
        print(filename) # This line will print the name of each GeoJSON file
        # The rest of your code to read the file and append to gdfs
        # gdf = gpd.read_file(file_path)
        # gdfs.append(gdf)

print("\nFinished listing GeoJSON files.")

updated_lulc_fiji.geojson

Finished listing GeoJSON files.


## Step 1.2: Configure STAC access and search parameters

In this block we are accessing the digital earth pacific catalog. This catalog housed all the earth observation dataset including
sentinel imagery (S2 GEOMAD) product in this case. Then we will use the Client.open function to open the catalog so that we can 
access it.

>catalog = "https://stac.digitalearthpacific.org"

>client = Client.open(catalog)

In [7]:
catalog = "https://stac.digitalearthpacific.org"

client = Client.open(catalog)

<font color='blue'>1.1. Reading the datapoints using the "read_file" function

Your code goes in the cells below. Add more cells here by clicking on the + button above.

First let's try and read your geojson datapoints in using the geopandas (gpd) library and the read_file function   

> training = gpd.read_file("training-data/ filename.geojson")

Second, we have to set the coordinate reference system to a suitable crs (e.g EPSG:4326)

>training = training.to_crs("EPSG:4326")

Define your bounding box by using the .total_bounds function to extract the lower_left furthest point coordinates and
the top right furthest point coordinates and convert then to a bounding box.
> min_lon, min_lat, max_lon, max_lat = training.total_bounds

Assign a new variable called bbox
>bbox = [min_lon, min_lat, max_lon, max_lat]


print the attributes in the dataset
> training

In [8]:
aoi = gpd.read_file("Nadroga_Navosa_Province.geojson")

geom = aoi.geometry.iloc[0].__geo_interface__ 

year = '2025'

items = list(
    client.search(
        collections=["dep_s2_geomad"], 
        datetime=year, 
        intersects=geom
    ).items()
)

print(f"Found {len(items)} items")


Found 1 items


In [87]:
data = load(
    items,
    measurements = ["nir", "red", "blue", "green", "emad", "smad", "bcmad", "green", "nir08", "nir09", "swir16", "swir22", "coastal", "rededge1", "rededge2", "rededge3"],
    intersects=aoi,
    resolution=10,
    chunks={"x": 2048, "y": 2048},
    groupby="solar_day",
)

data

<xarray.Dataset> Size: 998MB
Dimensions:      (y: 4017, x: 6901, time: 1)
Coordinates:
  * y            (y) float64 32kB -2.003e+06 -2.003e+06 ... -2.043e+06
  * x            (x) float64 55kB 3.058e+06 3.058e+06 ... 3.127e+06 3.127e+06
    spatial_ref  int32 4B 3832
  * time         (time) datetime64[ns] 8B 2025-01-01
Data variables: (12/15)
    nir          (time, y, x) uint16 55MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    red          (time, y, x) uint16 55MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    blue         (time, y, x) uint16 55MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    green        (time, y, x) uint16 55MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    emad         (time, y, x) float32 111MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    smad         (time, y, x) float32 111MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    ...           ...
    swir16       (time, y, x) uint16 55MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    swir22       (time, y, x) uint16 55MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    coastal      (time, y, x) uint16 55MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    rededge1     (time, y, x) uint16 55MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    rededge2     (time, y, x) uint16 55MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    rededge3     (time, y, x) uint16 55MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>

by running this block of code you can print the class code and the number of points for each classes.
> print(training['Class'].value_counts())

Also, print the total number of points in this dataset
> print('total gps points',(len(training)))

In this code you will search for dataset in the dep catalog using the client.search function and then define the 
collection or product you want to search, then redefine your datetime and bounding box. 

> datetime = "2023"
 
> items = client.search(
    collections=["dep_s2_geomad"],
    datetime=datetime,
    bbox=bbox
).item_collection()

Print the collection
>print(f"Found {len(items)} items in for {datetime}")


In [88]:
# aoi = gpd.read_file("Nadroga_Navosa_Province.geojson")

In [89]:
# # Force parameters into floats just to be safe
# bbox = [float(x) for x in bbox.flatten()] if hasattr(bbox, "flatten") else [float(x) for x in bbox]

# datetime = "2023"

# items = client.search(
#     collections=["dep_s2_geomad"],
#     datetime=datetime,
#     bbox=bbox,
#     method="GET"  # <-- Add this line to change the API request type
# ).item_collection()


In [90]:
# datetime = "2023"

# items = client.search(
#     collections=["dep_s2_geomad"],
#     datetime=datetime,
#     bbox=bbox
# ).item_collection()

Define all the bands in the satellite imagery you have accessed earlier. Load all the bands, the bounding box that define your area of interes
and the location of the collection (imagery)

> measurements = ["", "", ""]

> data = load_data(
    items,
    measurements,
    bbox,


> print(data)

In [91]:
# measurements = ["red", "green", "blue", "nir08"a]

# data = load_data(items, measurements, bbox)

# print(data)

This cell initializes a Dask client for parallel processing. The notebook allocates one worker with multiple threads and defines a memory limit to improve performance when working with large geospatial datasets. 

>dask_client = DaskClient(n_workers=1, threads_per_worker=16, memory_limit='16GB')

This cell also runs configure_s3_access() to establish secure access to cloud-hosted datasets stored in S3-compatible object storage.

>configure_s3_access(cloud_defaults=True, requester_pays=True)

This cell applies the scale() function to the loaded satellite dataset. Scaling converts raw pixel values into meaningful reflectance values that can be used for scientific analysis. 
>scaled = scale(data)

The .squeeze() method is then applied to remove unnecessary dimensions from the dataset and simplify its structure.
>scaled = scaled.squeeze() # .compute()

In [92]:
scaled = scale(data)
scaled = scaled.squeeze()

In [93]:
# bbox = 

In [94]:
# Explore the site we are working on
# scaled.odc.explore(vmin=0, vmax=0.3, bands=["red", "green", "blue"], crs="EPSG:3832", name=site)

This cell calculates additional spectral indices from the satellite imagery using the calculate_band_indices() function. Spectral indices are mathematical combinations of different spectral bands that highlight environmental properties such as vegetation health or water presence. 
>scaled = calculate_band_indices(scaled)

>scaled

These new layers provide additional information that can improve classification and analysis.

In [95]:
scaled = calculate_band_indices(scaled)

scaled

<xarray.Dataset> Size: 7GB
Dimensions:        (y: 4017, x: 6901)
Coordinates:
  * y              (y) float64 32kB -2.003e+06 -2.003e+06 ... -2.043e+06
  * x              (x) float64 55kB 3.058e+06 3.058e+06 ... 3.127e+06 3.127e+06
    spatial_ref    int32 4B 3832
    time           datetime64[ns] 8B 2025-01-01
Data variables: (12/32)
    nir            (y, x) float64 222MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    red            (y, x) float64 222MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    blue           (y, x) float64 222MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    green          (y, x) float64 222MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    emad           (y, x) float32 111MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    smad           (y, x) float32 111MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    ...             ...
    ndci           (y, x) float64 222MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    nbi            (y, x) float64 222MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    ndmi           (y, x) float64 222MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    bsi            (y, x) float64 222MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    awei           (y, x) float64 222MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    tc_wetness     (y, x) float64 222MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>

This cell applies the all_masks() function to the dataset. The masking process removes unwanted pixels such as clouds, cloud shadows, water, or invalid observations. The function also returns the generated mask itself so the excluded areas can be inspected if necessary.
>scaled, mask = all_masks(scaled, return_mask = True)

This line uses the .odc.eplore function to visualize the masked dataset
>'# mask.odc.explore(vmin=0, vmax=0.3, bands=["red", "green", "blue"], crs="EPSG:3832", name=site)'

In [96]:
# scaled
# scaled.odc.explore(vmin=0, vmax=0.3, bands=["red", "green", "blue"], crs="EPSG:3832", name=site)

### Postcard csv

The objective of this notebook was to train the machine learning model that will allow us to classify an area with land cover classes defined through the training data.

Step 1.2. Input the training data to sample geomad data from the postcard

his cell reprojects the training GeoDataFrame into the same coordinate reference system as the GeoMAD raster dataset. Reprojection is necessary to ensure spatial alignment between vector training points and raster imagery.
###### Reproject training data to the GeoMAD CRS and convert to xarray
>training_reprojected = training.to_crs(scaled.odc.crs)

>training_da = training_reprojected.assign(
>>x=training_reprojected.geometry.x, y=training_reprojected.geometry.y

>).to_xarray()


The geometry coordinates are then extracted into separate x and y columns to prepare the data for raster value extraction.
This cell converts the GeoDataFrame into an xarray object and extracts raster pixel values from the processed satellite dataset at each training point location. The nearest-neighbour method is used to match raster values to the point coordinates. The extracted values are then converted into a Pandas DataFrame for easier tabular analysis.
###### Extract training values from the masked dataset
>training_values = (
>>scaled.sel(training_da[["x", "y"]], method="nearest")
>.squeeze()
>.compute()
>>.topandas())

>training_values

This cell displays the column names of the extracted training dataset. This allows the user to inspect all available spectral bands, indices, and metadata variables that were extracted from the imagery.
###### Join the training data with the extracted values and remove unnecessary columns
>training_values.columns

This cell combines the extracted raster values with the corresponding land cover class labels stored in the ClassId column. The pd.concat() function merges the class labels and spectral variables into a single training array.
>training_array = pd.concat([training["ClassId"], training_values], axis=1)

This cell removes rows containing missing values using the dropna() function. Removing incomplete observations ensures that only valid training samples remain in the dataset before machine learning analysis.
###### Drop rows where there was no data available
training_array = training_array.dropna()

This cell converts the ClassId column into integer format using .astype(int). Converting class labels into integers is important because machine learning algorithms typically require numerical class identifiers.
###### Convert LULC_code to integer
>training_array["ClassId"] = training_array["ClassId"].astype(int)

This cell displays the first few rows of the completed training array using .head(). This allows the user to preview the dataset and verify that the extracted variables and class labels were combined correctly.
>training_array.head()

This cell displays the column names of the extracted training dataset. This allows the user to inspect all available spectral bands, indices, and metadata variables that were extracted from the imagery.
>print(training_array.shape[1], 'total columns')

>print('columns included', training_array.columns)

This cell counts the number of samples belonging to each class using value_counts(). The output helps evaluate whether the training dataset is balanced across land cover categories. The total number of GPS training points is also displayed.
>print(training_array['ClassId'].value_counts())

>print('total gps points',(len(training_array)))

This cell removes unnecessary metadata columns such as spatial_ref and time from the training dataset. These variables are not required for machine learning and are removed to simplify the final output.
>training_array=training_array.drop(columns=["spatial_ref", "time"])

This cell displays the cleaned training dataset after unnecessary metadata fields have been removed. This allows the user to inspect the final structure before exporting the dataset.
>training_array

This cell exports the processed training array as a CSV file into the training-data/ directory. The output filename uses the version string created earlier in the notebook, ensuring that the exported file has a unique and traceable name.
>training_array.to_csv(f"training-data/{version}-training.csv", index=False)

This cell checks the data type of the ClassId column. The purpose is to confirm that the class labels were successfully converted into integer format.
>training_array["ClassId"].dtype

This final cell prints the class label counts again after all cleaning and processing steps have been completed. This provides a final verification of the dataset composition before the workflow ends.
>print(training_array['ClassId'].value_counts())